In [ ]:
# libraries
import os
import json
import time
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import nbformat
from IPython.display import display

In [ ]:
# globals; run through run_pipeline.ipynb
ROOT = Path(os.environ['NCA_PROJECT_ROOT'])
TASK = ROOT / 'e_phase1_test'
INPUT, OUTPUT = TASK / 'input', TASK / 'output'
CFG = json.loads((INPUT / 'config.json').read_text())
DEVICE = torch.device(CFG['device'])
torch.set_num_threads(CFG['cpu_threads'])
torch.use_deterministic_algorithms(True)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = False
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

In [ ]:
# functions passed through c_models/output; definitions only, no training side effects
runtime_nb = nbformat.read(INPUT / 'runtime.ipynb', as_version=4)
for runtime_cell in runtime_nb.cells:
    exec(compile(runtime_cell.source, 'runtime.ipynb', 'exec'), globals())
PACK = torch.load(INPUT / 'dataset.pt', weights_only=True, map_location='cpu')
SPECS = CFG['models']

In [ ]:
if CFG.get('task') not in ['canonical_reconstruction', 'communication_study']:
    # run and condition imports
    manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')
    conditions = pd.read_csv(INPUT / 'conditions.csv')
    metric_frames, dynamics_frames, channel_rows, equivariance_rows, examples = [], [], [], [], []

In [ ]:
if CFG.get('task') not in ['canonical_reconstruction', 'communication_study']:
    # held-out reconstruction and external transmission errors
    with torch.no_grad():
        for run in manifest.to_dict('records'):
            spec = next(item for item in SPECS if item['name'] == run['model'])
            model, checkpoint = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
            generator = torch_stream(CFG, DEVICE, 50 + run['replicate'])
            metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
            steps = steps_for(model, run['regime'], CFG['eval_steps'])
            for split in ['test_composition', 'test_angle', 'test_color', 'test_translation', 'test_joint']:
                ids = panel_ids(PACK, split, CFG)
                p, u = np.zeros(len(ids)), np.full(len(ids), 0.5)
                target, _ = select_target(PACK, ids, p, u, DEVICE)
                for rate in CFG['stress_flip_rates']:
                    for trial in range(CFG['channel_trials']):
                        started = time.perf_counter()
                        state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, flips=all_flips(rate))
                        if DEVICE.type == 'cuda':
                            torch.cuda.synchronize()
                        row_metadata = {**metadata, 'split': split, 'condition': ids, 'flip_rate': rate, 'trial': trial,
                                        'intervention': 'none', 'steps': steps, 'bits': bit_cost(model, steps),
                                        'seconds_per_image': (time.perf_counter() - started) / len(ids)}
                        metric_frames.append(metric_rows(state, target, row_metadata))
                        if rate == 0 and trial == 0:
                            examples.append({**metadata, 'split': split, 'condition': int(ids[0]),
                                             'target': target[0].cpu(), 'generated': state[0, ..., :4].cpu()})
            display(pd.DataFrame([{**metadata, 'test': 'reconstruction and error sweep', 'status': 'complete'}]))

In [ ]:
if CFG.get('task') not in ['canonical_reconstruction', 'communication_study']:
    # channel interventions, persistence, repair and missing-seed control
    with torch.no_grad():
        for run in manifest.to_dict('records'):
            spec = next(item for item in SPECS if item['name'] == run['model'])
            model, _ = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
            generator = torch_stream(CFG, DEVICE, 60 + run['replicate'])
            metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
            ids = panel_ids(PACK, 'test_composition', CFG)[:CFG['batch_size']]
            p, u = np.zeros(len(ids)), np.full(len(ids), 0.5)
            target, _ = select_target(PACK, ids, p, u, DEVICE)
            cue = make_cue(PACK, ids, p, u, CFG, DEVICE)
            steps = steps_for(model, run['regime'], CFG['eval_steps'])
            state, frames = run_steps(model, seed_state(PACK, ids, CFG, DEVICE), cue, 2 * steps, generator,
                                      record=sorted(set([max(1, steps // 2), steps, steps + steps // 2, 2 * steps])))
            for step, frame in frames.items():
                dynamics_frames.append(metric_rows(frame.to(DEVICE), target, {**metadata, 'condition': ids,
                                       'intervention': 'persistence', 'steps': step, 'bits': bit_cost(model, step)}))
            grown = frames[steps].to(DEVICE)
            channel_rows.extend(channel_diagnostics(model, grown, cue, generator, metadata))
            damaged = centered_damage(grown, PACK['coordinates'][ids], CFG)
            dynamics_frames.append(metric_rows(damaged, target, {**metadata, 'condition': ids, 'intervention': 'damage_initial',
                                   'steps': 0, 'bits': 0}))
            repaired, _ = run_steps(model, damaged, cue, steps, generator)
            dynamics_frames.append(metric_rows(repaired, target, {**metadata, 'condition': ids, 'intervention': 'repair',
                                   'steps': steps, 'bits': bit_cost(model, steps)}))
            channels = [('m',)] if spec['family'] == 'message' else [('q',), ('r',), ('q', 'r')]
            for muted in channels:
                muted_state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, mute=muted)
                metric_frames.append(metric_rows(muted_state, target, {**metadata, 'condition': ids, 'split': 'test_composition',
                                     'flip_rate': 0.0, 'trial': 0, 'intervention': 'mute_' + ''.join(muted), 'steps': steps,
                                     'bits': bit_cost(model, steps)}))
            blank_state = generate_panel(model, PACK, CFG, ids, p, u, run['regime'], generator, blank=True)
            metric_frames.append(metric_rows(blank_state, target, {**metadata, 'condition': ids, 'split': 'test_composition',
                                 'flip_rate': 0.0, 'trial': 0, 'intervention': 'remove_seed_and_cue', 'steps': steps,
                                 'bits': bit_cost(model, steps)}))

In [ ]:
if CFG.get('task') not in ['canonical_reconstruction', 'communication_study']:
    # transformation consistency; score both consistency and target fit so blank outputs cannot look successful
    base_id = int(conditions.query('angle == 0 and dx == 0 and dy == 0 and saturation == 1 and red_gain == 1')['condition'].iloc[0])
    with torch.no_grad():
        for run in manifest.to_dict('records'):
            spec = next(item for item in SPECS if item['name'] == run['model'])
            model, _ = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
            generator = torch_stream(CFG, DEVICE, 70 + run['replicate'])
            metadata = {key: run[key] for key in ['model', 'regime', 'replicate']}
            canonical = generate_panel(model, PACK, CFG, [base_id], [0.0], [0.5], run['regime'], generator)[..., :4].clamp(0, 1)
            for condition in panel_ids(PACK, 'test_angle', CFG)[:3]:
                row = conditions.iloc[int(condition)]
                state = generate_panel(model, PACK, CFG, [condition], [0.0], [0.5], run['regime'], generator)
                expected = transform_rgba(canonical, row['angle'], row['dx'], row['dy'], row['saturation'],
                                           [row['red_gain'], row['green_gain'], row['blue_gain']])
                target = PACK['targets'][condition, 0].float().to(DEVICE)[None]
                equivariance_rows.append({**metadata, 'condition': int(condition), 'angle': row['angle'],
                                          'consistency_mse': float((state[..., :4].clamp(0, 1) - expected).square().mean()),
                                          'target_mse': float(image_metrics(state, target)['mse'][0])})

In [ ]:
if CFG.get('task') not in ['canonical_reconstruction', 'communication_study']:
    # exports
    pd.concat(metric_frames, ignore_index=True).to_csv(OUTPUT / 'phase1_metrics.csv', index=False)
    pd.concat(dynamics_frames, ignore_index=True).to_csv(OUTPUT / 'phase1_dynamics.csv', index=False)
    pd.DataFrame(channel_rows).to_csv(OUTPUT / 'phase1_channels.csv', index=False)
    pd.DataFrame(equivariance_rows).to_csv(OUTPUT / 'phase1_transform_consistency.csv', index=False)
    torch.save(examples, OUTPUT / 'phase1_examples.pt')
    display(pd.concat(metric_frames).query("intervention == 'none' and flip_rate == 0").groupby(['model', 'regime', 'split'])[
            ['mse', 'foreground_rmse', 'alpha_iou']].mean().reset_index())

In [ ]:
# fixed-image diagnostic: assess detail, persistence and dependence on communication
if CFG.get('task') == 'canonical_reconstruction':
    manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')
    condition = PACK['splits']['test_reconstruction'][0]
    target = PACK['targets'][condition:condition + 1, 0].float().to(DEVICE)
    flat = target.clone()
    alpha = target[..., 3:4]
    average = target[..., :3].sum((1, 2), keepdim=True) / alpha.sum((1, 2), keepdim=True)
    flat[..., :3] = average * alpha
    flat_detail = float(detail_error(flat, target).sqrt()[0])
    frames, metric_frames, baseline_frames, diagnostic_channels = [], [], [], []
    baseline_frames.append(metric_rows(torch.zeros_like(target), target, {'baseline': 'blank'}))
    baseline_frames.append(metric_rows(flat, target, {'baseline': 'flat_color_with_oracle_target_mask'}))
    for run in manifest.to_dict('records'):
        spec = next(s for s in SPECS if s['name'] == run['model'])
        model, checkpoint = load_trained(INPUT / 'phase1_runs' / run['checkpoint'], spec, CFG, DEVICE, run['replicate'])
        generator = torch_stream(CFG, DEVICE, 50 + run['replicate'])
        cue = make_cue(PACK, [condition], [0.0], [0.5], CFG, DEVICE)
        state = seed_state(PACK, [condition], CFG, DEVICE)
        horizon = CFG['eval_steps']
        state, snapshots = run_steps(model, state, cue, 2 * horizon, generator, record=[horizon, 2 * horizon])
        metadata = {k: run[k] for k in ['model', 'regime', 'replicate']}
        for step, snapshot in snapshots.items():
            result = metric_rows(snapshot.to(DEVICE), target, {**metadata, 'steps': step, 'intervention': 'none'})
            result['detail_ratio_to_flat'] = result['detail_rmse'] / flat_detail
            result['quality_pass'] = (result['foreground_rmse'].le(CFG['gate_foreground_rmse'])
                                      & result['alpha_iou'].ge(CFG['gate_alpha_iou'])
                                      & result['detail_ratio_to_flat'].le(CFG['gate_detail_ratio']))
            metric_frames.append(result)
        frames.append({**metadata, 'generated': snapshots[horizon][0, ..., :4],
                       'persistent': snapshots[2 * horizon][0, ..., :4], 'best_step': checkpoint['step']})
        grown = snapshots[horizon].to(DEVICE)
        diagnostic_channels.extend(channel_diagnostics(model, grown, cue, generator, metadata))
        mute = ('continuous',) if spec['family'] == 'continuous' else ('m',) if spec['family'] == 'message' else ('q', 'r')
        muted, _ = run_steps(model, seed_state(PACK, [condition], CFG, DEVICE), cue, horizon, generator, mute=mute)
        metric_frames.append(metric_rows(muted, target, {**metadata, 'steps': horizon, 'intervention': 'mute_all'}))
        blank, _ = run_steps(model, torch.zeros_like(state), torch.zeros_like(cue), horizon, generator)
        assert torch.equal(blank, torch.zeros_like(blank)), 'dormant grid grew without a cue or message'
    metrics = pd.concat(metric_frames, ignore_index=True)
    summary = metrics.query("intervention == 'none'").copy()
    summary['quality_pass'] = summary['quality_pass'].astype(bool)
    metrics.to_csv(OUTPUT / 'reconstruction_metrics.csv', index=False)
    summary.to_csv(OUTPUT / 'reconstruction_summary.csv', index=False)
    pd.concat(baseline_frames).to_csv(OUTPUT / 'reconstruction_baselines.csv', index=False)
    pd.DataFrame(diagnostic_channels).to_csv(OUTPUT / 'reconstruction_channels.csv', index=False)
    gate = {'scope': 'same-template reconstruction and persistence, not semantic or transformation generalization',
            'foreground_rmse_max': CFG['gate_foreground_rmse'], 'alpha_iou_min': CFG['gate_alpha_iou'],
            'detail_ratio_max': CFG['gate_detail_ratio'], 'flat_baseline_uses_oracle_mask': True,
            'passed_models_at_eval_horizon': summary.loc[summary['steps'].eq(horizon) & summary['quality_pass'], 'model'].tolist(),
            'passed_models_at_double_horizon': summary.loc[summary['steps'].eq(2 * horizon) & summary['quality_pass'], 'model'].tolist()}
    (OUTPUT / 'reconstruction_gate.json').write_text(json.dumps(gate, indent=2))
    torch.save({'target': target.cpu(), 'flat_baseline': flat.cpu(), 'examples': frames}, OUTPUT / 'reconstruction_examples.pt')
    display(summary[['model', 'steps', 'foreground_rmse', 'detail_rmse', 'alpha_iou', 'detail_ratio_to_flat', 'quality_pass']])

In [ ]:
# the figure includes a featureless baseline; good mask overlap alone cannot hide missing facial details
if CFG.get('task') == 'canonical_reconstruction':
    import matplotlib.pyplot as plt
    import seaborn as sns
    sns.set_theme(style='whitegrid', palette='colorblind')
    def composite(rgba):
        rgba = rgba.detach().cpu().clamp(0, 1)
        return (rgba[..., :3] + 1 - rgba[..., 3:4]).clamp(0, 1).numpy()
    fig, axes = plt.subplots(3, len(frames) + 1, figsize=(3 * (len(frames) + 1), 8), squeeze=False)
    for column, example in enumerate(frames):
        for row, rgba in enumerate([target[0].cpu(), example['generated'], example['persistent']]):
            axes[row, column].imshow(composite(rgba))
        axes[0, column].set_title(example['model'])
    for row in range(3):
        axes[row, -1].imshow(composite(target[0] if row == 0 else flat[0]))
    axes[0, -1].set_title('flat color / oracle mask')
    for row, label in enumerate(['target', f'generated: {horizon} steps', f'persistent: {2 * horizon} steps']):
        axes[row, 0].set_ylabel(label)
    for ax in axes.flat:
        ax.set_xticks([])
        ax.set_yticks([])
    fig.suptitle('Fixed-image reconstruction diagnostic; no transformation/generalization claim')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'reconstruction_examples.png', dpi=170)
    plt.show()
    histories = []
    for run in manifest.to_dict('records'):
        history = pd.read_csv(INPUT / 'phase1_runs' / Path(run['checkpoint']).parent / 'history.csv')
        history['model'] = run['model']
        histories.append(history)
    histories = pd.concat(histories, ignore_index=True)
    histories.to_csv(OUTPUT / 'reconstruction_training_history.csv', index=False)
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.lineplot(data=histories, x='step', y='val_loss', hue='model', style='model', markers=True, errorbar=None, ax=ax)
    ax.set_ylabel('validation pixel + interior-detail loss')
    fig.tight_layout()
    fig.savefig(OUTPUT / 'reconstruction_learning.png', dpi=170)
    plt.show()

In [ ]:
# independent evaluation workers; one output directory per immutable training-row identity
if CFG.get('task') == 'communication_study':
    import hashlib
    jobs = pd.read_csv(ROOT / 'job_matrix.csv')
    selected = os.environ.get('NCA_RUN_INDEX', '')
    if selected:
        jobs = jobs.loc[jobs['index'].eq(int(selected))]
    manifest = pd.read_csv(INPUT / 'phase1_manifest.csv')
    conditions = pd.read_csv(INPUT / 'conditions.csv')
    evaluation_root = TASK / 'output' / 'eval_runs'
    evaluation_root.mkdir(exist_ok=True)
    for job in jobs.to_dict('records'):
        folder = evaluation_root / job['run']
        folder.mkdir(exist_ok=True)
        with (folder / '.run.lock').open('a') as handle:
            try:
                fcntl.flock(handle, fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError(f'evaluation already has an active writer: {folder}') from None
            run = manifest.loc[manifest['model'].eq(job['model']) & manifest['regime'].eq(job['regime']) & manifest['replicate'].eq(job['replicate'])].iloc[0]
            spec = next(s for s in SPECS if s['name'] == job['model'])
            checkpoint_path = INPUT / 'phase1_runs' / run['checkpoint']
            checkpoint_hash = hashlib.sha256(checkpoint_path.read_bytes()).hexdigest()
            completed = folder / 'completion.json'
            if completed.exists():
                if os.environ.get('NCA_RESUME', '0') != '1':
                    raise FileExistsError(f'evaluation exists: {folder}; use explicit resume')
                old = json.loads(completed.read_text())
                if old['training_signature'] != run['signature'] or old['checkpoint_sha256'] != checkpoint_hash:
                    raise ValueError('evaluation checkpoint identity changed')
                if all(hashlib.sha256((folder / name).read_bytes()).hexdigest() == digest for name, digest in old['files'].items()):
                    continue
            model, checkpoint = load_trained(checkpoint_path, spec, CFG, DEVICE, job['replicate'])
            metadata = study_run_metadata(spec, job['regime'], job['replicate'])
            metadata.update({f'train_flip_{ch}': spec['train_flips'].get(ch, 0.0) for ch in ['m', 'q', 'r']})
            scopes = ['m'] if spec['family'] == 'message' else ['q', 'r', 'qr']
            scenarios = [('none', 0.0, {})]
            if spec['kind'] == 'binary':
                scenarios += [(scope, rate, {ch: rate for ch in scope}) for rate in CFG['stress_flip_rates'] if rate > 0 for scope in scopes]
            metric_frames, examples, consistency_rows = [], [], []
            with torch.no_grad():
                for split in ['train', 'test_composition', 'test_angle', 'test_color', 'test_translation', 'test_joint']:
                    ids = panel_ids(PACK, split, CFG)
                    target = PACK['targets'][ids, 0].float().to(DEVICE)
                    for scope, rate, flips in scenarios:
                        for trial in range(CFG['channel_trials']):
                            generator = torch_stream(CFG, DEVICE, 50 + job['replicate'] * CFG['channel_trials'] + trial)
                            # same stream start for each intervention isolates channel changes within a trained model
                            states = {multiple: [] for multiple in CFG['validation_horizons']}
                            for start in range(0, len(ids), CFG['batch_size']):
                                batch_ids = ids[start:start + CFG['batch_size']]
                                cue = make_cue(PACK, batch_ids, np.zeros(len(batch_ids)), np.full(len(batch_ids), 0.5), CFG, DEVICE)
                                state = seed_state(PACK, batch_ids, CFG, DEVICE)
                                previous = 0
                                for multiple in CFG['validation_horizons']:
                                    horizon = steps_for(model, job['regime'], multiple * CFG['eval_steps'])
                                    state, _ = run_steps(model, state, cue, horizon - previous, generator, flips=flips)
                                    states[multiple].append(state.cpu())
                                    previous = horizon
                            for multiple, pieces in states.items():
                                state = torch.cat(pieces).to(DEVICE)
                                values = study_quality(state, target, CFG)
                                frame = pd.DataFrame({key: value.cpu().numpy() for key, value in values.items()})
                                horizon = steps_for(model, job['regime'], multiple * CFG['eval_steps'])
                                for key, value in {**metadata, 'split': split, 'condition': ids, 'trial': trial, 'test_scope': scope,
                                                   'flip_rate': rate, 'horizon_multiple': multiple, 'steps': horizon,
                                                   'bits_per_image': bit_cost(model, horizon), 'cell_updates_per_image': horizon * CFG['grid'] ** 2}.items():
                                    frame[key] = value
                                yy, xx = torch.meshgrid(torch.arange(CFG['grid'], device=DEVICE), torch.arange(CFG['grid'], device=DEVICE), indexing='ij')
                                seed_xy = PACK['coordinates'][ids].to(DEVICE)
                                distance = torch.maximum((yy[None] - seed_xy[:, 0, None, None]).abs(), (xx[None] - seed_xy[:, 1, None, None]).abs())
                                outside = distance > horizon
                                alpha = target[..., 3]
                                frame['target_fraction_outside_seed_cone'] = ((alpha * outside).sum((1, 2)) / alpha.sum((1, 2))).cpu().numpy()
                                lower_bound = ((target.square().sum(-1) * alpha * outside).sum((1, 2)) / (4 * alpha.sum((1, 2)))).sqrt()
                                frame['clean_seed_cone_fg_rmse_bound'] = lower_bound.cpu().numpy() if rate == 0 else np.nan
                                if spec.get('cue_mode') == 'broadcast':
                                    frame['target_fraction_outside_seed_cone'] = np.nan
                                    frame['clean_seed_cone_fg_rmse_bound'] = np.nan
                                metric_frames.append(frame)
                                if scope == 'none' and trial == 0 and multiple == 1:
                                    examples.append({'split': split, 'condition': int(ids[0]), 'target': target[0].cpu(), 'generated': state[0, ..., :4].cpu()})
                # paired transformation consistency under clean external transmission; target error is retained alongside it
                base_id = int(conditions.query('angle == 0 and dx == 0 and dy == 0 and saturation == 1 and red_gain == 1')['condition'].iloc[0])
                generator = torch_stream(CFG, DEVICE, 70 + job['replicate'])
                canonical = generate_panel(model, PACK, CFG, [base_id], [0.0], [0.5], job['regime'], generator, flips={})[..., :4].clamp(0, 1)
                for condition in panel_ids(PACK, 'test_joint', CFG)[:3]:
                    row = conditions.iloc[int(condition)]
                    generator = torch_stream(CFG, DEVICE, 70 + job['replicate'])
                    state = generate_panel(model, PACK, CFG, [condition], [0.0], [0.5], job['regime'], generator, flips={})
                    expected = transform_rgba(canonical, row['angle'], row['dx'], row['dy'], row['saturation'], [row['red_gain'], row['green_gain'], row['blue_gain']])
                    target = PACK['targets'][condition:condition + 1, 0].float().to(DEVICE)
                    consistency_rows.append({**metadata, 'condition': int(condition), 'consistency_mse': float((state[..., :4].clamp(0, 1) - expected).square().mean()),
                                             'target_mse': float(image_metrics(state, target)['mse'][0])})
                # paired cue intervention; identical initial state, only angle or saturation differs
                if CFG.get('conditioning_diagnostic', False) or CFG.get('quantization_study', False) or CFG.get('reliability_study', False):
                    pair_rows, pair_examples = [], []
                    for cue_trial in range(CFG['channel_trials'] if CFG.get('reliability_study', False) else 1):
                        for pair in conditioning_pairs(conditions, CFG['eval_cases']):
                            pair_ids = [pair['a'], pair['b']]
                            cue = make_cue(PACK, pair_ids, [0, 0], [0.5, 0.5], CFG, DEVICE)
                            target, _ = select_target(PACK, pair_ids, [0, 0], [0.5, 0.5], DEVICE)
                            state = seed_state(PACK, pair_ids, CFG, DEVICE)
                            audit_condition_batch(PACK, np.array(pair_ids), np.array(pair_ids), cue, target, CFG)
                            assert torch.equal(state[0], state[1]), 'paired initial states differ'
                            generator = torch_stream(CFG, DEVICE, 80 + job['replicate'] * CFG['channel_trials'] + cue_trial if CFG.get('reliability_study', False) else 80 + job['replicate'])
                            previous = 0
                            for multiple in CFG['validation_horizons']:
                                horizon = multiple * CFG['eval_steps']
                                if CFG.get('reliability_study', False):
                                    state = run_paired_cue_steps(model, state, cue, horizon - previous, generator)
                                else:
                                    state, _ = run_steps(model, state, cue, horizon - previous, generator, flips={})
                                previous = horizon
                                rgba = state[..., :4].clamp(0, 1)
                                correct = (rgba - target).square().mean().item()
                                swapped = (rgba - target.flip(0)).square().mean().item()
                                target_change = (target[0] - target[1]).square().mean().item()
                                output_change = (rgba[0] - rgba[1]).square().mean().item()
                                pair_rows.append({**metadata, **pair, **({'trial': cue_trial, 'noise_coupling': 'common_random_numbers'} if CFG.get('reliability_study', False) else {}), 'cue_mode': spec['cue_mode'], 'horizon_multiple': multiple,
                                                  'output_change_mse': output_change, 'target_change_mse': target_change,
                                                  'response_ratio': output_change / target_change, 'correct_target_mse': correct,
                                                  'swapped_target_mse': swapped, 'cue_match_advantage': swapped - correct})
                                if multiple == 1 and not any(x['domain'] == pair['domain'] and x['factor'] == pair['factor'] for x in pair_examples):
                                    pair_examples.append({**pair, 'target': target.cpu(), 'generated': rgba.cpu()})
                    pd.DataFrame(pair_rows).to_csv(folder / 'cue_response.csv', index=False)
                    torch.save(pair_examples, folder / 'cue_examples.pt')
                # channel use and mute tests on identical held-out inputs; no damage/adaptation tests
                ids = panel_ids(PACK, 'test_composition', CFG)[:1]
                cue = make_cue(PACK, ids, [0.0], [0.5], CFG, DEVICE)
                target = PACK['targets'][ids, 0].float().to(DEVICE)
                horizon = steps_for(model, job['regime'], CFG['eval_steps'])
                state, _ = run_steps(model, seed_state(PACK, ids, CFG, DEVICE), cue, horizon, generator, flips={})
                channel_rows = channel_diagnostics(model, state, cue, generator, metadata, flips={}) if spec['kind'] == 'binary' else []
                muted = ('continuous',) if spec['kind'] == 'continuous' else ('m',) if spec['family'] == 'message' else ('q', 'r')
                ablations = []
                for label, no_seed, mute in [('mute_all', False, muted), ('no_seed_or_cue', True, ())]:
                    result = generate_panel(model, PACK, CFG, ids, [0.0], [0.5], job['regime'], generator, flips={}, mute=mute, blank=no_seed)
                    ablations.append(metric_rows(result, target, {**metadata, 'intervention': label}))
            metrics = pd.concat(metric_frames, ignore_index=True)
            groups = list(metadata) + ['split', 'test_scope', 'flip_rate', 'horizon_multiple', 'steps', 'bits_per_image', 'cell_updates_per_image']
            score_columns = ['mse', 'raw_mse', 'foreground_rmse', 'alpha_iou', 'detail_rmse', 'detail_ratio_to_flat', 'quality_pass', 'target_fraction_outside_seed_cone', 'clean_seed_cone_fg_rmse_bound']
            summary = metrics.groupby(groups, dropna=False, sort=False)[score_columns].mean().reset_index()
            summary = summary.rename(columns={'quality_pass': 'success_fraction'})
            metrics.to_csv(folder / 'case_metrics.csv', index=False)
            summary.to_csv(folder / 'replicate_scores.csv', index=False)
            pd.DataFrame(consistency_rows).to_csv(folder / 'transform_consistency.csv', index=False)
            pd.DataFrame(channel_rows, columns=list(metadata) + ['channel', 'one_rate', 'empirical_binary_entropy', 'mean_sampling_variance', 'near_saturation_fraction']).to_csv(folder / 'channels.csv', index=False)
            pd.concat(ablations).to_csv(folder / 'ablations.csv', index=False)
            torch.save(examples, folder / 'examples.pt')
            files = {name: hashlib.sha256((folder / name).read_bytes()).hexdigest() for name in ['case_metrics.csv', 'replicate_scores.csv', 'transform_consistency.csv', 'channels.csv', 'ablations.csv', 'examples.pt']}
            if CFG.get('conditioning_diagnostic', False) or CFG.get('quantization_study', False) or CFG.get('reliability_study', False):
                files.update({name: hashlib.sha256((folder / name).read_bytes()).hexdigest() for name in ['cue_response.csv', 'cue_examples.pt']})
            atomic_json({'index': job['index'], 'run': job['run'], 'training_signature': run['signature'],
                         'checkpoint_sha256': checkpoint_hash, 'files': files, 'case_rows': len(metrics)}, completed)
            print(json.dumps({'evaluation_run': job['run'], 'case_rows': len(metrics), 'complete': True}), file=sys.__stdout__, flush=True)
